# Notebook 02 — Event Definition & Detection
**Thesis:** Chapter 6 | **Input:** `data/1_reconstructed.csv` | **Output:** `data/2_events.csv`

---

## Purpose
Adds 18 event indicator columns to the reconstructed dataset.

An **event** is a binary or categorical condition at the time of each received packet.  
Example: "Was CO₂ above 700 ppm at this moment?"

| Family | Events | Why it matters |
|--------|--------|---------------|
| Occupancy (CO₂) | E1, E2, E3 | CO₂ is the best non-invasive proxy for human presence |
| Temporal | E4, E5, E6 | Office activity follows predictable daily and weekly cycles |
| Air Quality (PM2.5) | E7, E8 | PM2.5 spikes mark activity bursts — cleaning, printing, cooking |
| Atmospheric | E9, E10, E11, E12 | Pressure, humidity, temperature vary seasonally with HVAC |
| TX Config | E13, E14, E15 | SF controls Time-on-Air and therefore collision probability |
| Burst Loss | E16 | Classifies loss episodes by severity |
| Signal Context | E17, E18 | RSSI/ESP of last received packet — proxy for channel state |

Events are added to the full dataset.  
Notebook 03 filters to `PDR_link` intervals (excluding outages and SF artifacts) before correlating.

## 0 · Imports & Load

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / '1_reconstructed.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values(['device_id', 'time']).reset_index(drop=True)

print(f'Rows    : {len(df):,}')
print(f'Devices : {sorted(df["device_id"].unique())}')

Rows    : 1,217,313
Devices : ['ED0', 'ED1', 'ED2', 'ED3', 'ED4', 'ED5']


## 1 · Temporal Events  *(§6.2)*

Converted to `Europe/Berlin` local time — handles both DST transitions  
(October 2024 and March 2025) automatically.

| Event | Definition |
|-------|-----------|
| E2 | Weekday (1) vs weekend (0) — coarsest occupancy signal |
| E4 | Time-of-day band: night / morning / peak / evening |
| E5 | Office hours — weekday AND 08:00–17:59 |
| E6 | Season — autumn / winter / spring across 34-week campaign |

In [3]:
# local time — required for correct hour/weekday extraction across DST boundaries
local = df['time'].dt.tz_convert('Europe/Berlin')
df['hour']      = local.dt.hour
df['dayofweek'] = local.dt.dayofweek   # 0=Monday, 6=Sunday
df['month']     = local.dt.month

#todo Switch position of E2 and E1 so that we do occupancy first as E1 and E2, then the weekday shit starts in temporal Event as E3
#todo Switch Occupancy and temporal stuff
#todo ALSO NOTE THAT THE TEMPORAL TO SOME EXTENT IS RELATED TO OCCUPANCY!!!!
# E2: weekday vs weekend
df['e2_is_weekday'] = (df['dayofweek'] < 5).astype(int)

# E4: time-of-day bands — bins use 24 (not 23) to capture hour=23 correctly
df['e4_time_of_day'] = pd.cut(
    df['hour'],
    bins=[-1, 6, 9, 17, 24],
    labels=['night', 'morning', 'peak', 'evening']
)

# TODO Night: 00:00 – 06:59 → 7 hours
# TODO  Morning: 07:00 – 09:59 → 3 hours
# TODO  Peak: 10:00 – 17:59 → 8 hours
# TODO  Evening: 18:00 – 23:59 → 6 hours;  BALANCE THIS MORE!!!

# E5: office hours — weekday AND 08:00–17:59
df['e5_office_hours'] = (
    (df['e2_is_weekday'] == 1) & df['hour'].between(8, 17)
).astype(int)

# E6: season — campaign spans autumn 2024, winter 2024/25, spring 2025
df['e6_season'] = df['month'].map({
     9: 'autumn', 10: 'autumn', 11: 'autumn',
    12: 'winter',  1: 'winter',  2: 'winter',
     3: 'spring',  4: 'spring',  5: 'spring'
})

print(f'Weekday share : {df["e2_is_weekday"].mean()*100:.1f}%')
print(f'Office hours  : {df["e5_office_hours"].mean()*100:.1f}%')
print(f'Season counts : {df["e6_season"].value_counts().to_dict()}')

Weekday share : 70.9%
Office hours  : 29.2%
Season counts : {'spring': 443050, 'winter': 428825, 'autumn': 345438}


## 2 · Occupancy Events — CO₂  *(§6.1)*

CO₂ rises linearly with the number of people present — human metabolism produces ~200 ml CO₂/min at rest.  
Outdoor ambient is ~420 ppm. Weekday mean in this building is ~576 ppm vs ~457 ppm on weekends,  
confirming CO₂ as a reliable occupancy proxy.

| Event | Definition |
|-------|-----------|
| E1 | CO₂ tier: background (<500) / moderate (500–700) / high (>700 ppm) |
| E3 | CO₂ rising: delta > 20 ppm per 60s — occupancy actively increasing |

In [4]:
# E1: CO2 tier — 500 ppm = outdoor ambient ceiling, 700 ppm = moderate occupancy onset
df['e1_co2_tier'] = pd.cut(
    df['co2'],
    bins=[0, 500, 700, 10_000],
    labels=['background', 'moderate', 'high']
)

# E3: CO2 rising — delta > 20 ppm/60s signals people arriving
# 20 ppm chosen to exceed BME280 sensor noise (~±10 ppm)
# computed per device to avoid cross-device boundary artifacts
df['e3_co2_rising'] = (
    df.groupby('device_id')['co2'].diff() > 20
).astype(int)

print('CO2 tier distribution:')
print(df['e1_co2_tier'].value_counts(normalize=True).mul(100).round(1).astype(str).add('%').to_string())
print(f'\nWeekday CO2 mean : {df[df["e2_is_weekday"]==1]["co2"].mean():.0f} ppm')
print(f'Weekend CO2 mean : {df[df["e2_is_weekday"]==0]["co2"].mean():.0f} ppm')

CO2 tier distribution:
e1_co2_tier
background    51.7%
moderate      36.0%
high          12.3%

Weekday CO2 mean : 576 ppm
Weekend CO2 mean : 457 ppm


## 3 · Air Quality Events — PM2.5  *(§6.3)*

PM2.5 correlates with human activity bursts — cleaning, printing, cooking —  
rather than directly attenuating 868 MHz signals.

Device-specific 90th percentile thresholds for E7 account for location differences:  
ED4 (server room) baseline = 0.79 µg/m³ vs ED5 (kitchen) baseline = 6.28 µg/m³.  
A global threshold would over-flag ED5 and under-flag ED4.

| Event | Definition |
|-------|-----------|
| E7 | PM2.5 spike: above device-specific 90th percentile |
| E8 | PM2.5 absolute tier: clean (<2) / moderate (2–10) / elevated (>10 µg/m³) |

In [5]:
# E7: device-specific PM2.5 spike — 90th percentile per device
pm25_q90 = df.groupby('device_id')['pm25'].transform(lambda x: x.quantile(0.90))
df['e7_pm25_spike'] = (df['pm25'] > pm25_q90).astype(int)

# E8: absolute WHO-aligned tier
df['e8_pm25_tier'] = pd.cut(
    df['pm25'],
    bins=[-0.01, 2, 10, 10_000],
    labels=['clean', 'moderate', 'elevated']
)

print('Device-specific PM2.5 90th percentile thresholds:')
print(df.groupby('device_id')['pm25'].quantile(0.90).round(2).to_string())
print(f'\nSpike rate (E7=1): {df["e7_pm25_spike"].mean()*100:.1f}%')

Device-specific PM2.5 90th percentile thresholds:
device_id
ED0    3.86
ED1    4.02
ED2    3.46
ED3    4.05
ED4    0.79
ED5    6.28

Spike rate (E7=1): 10.0%


## 4 · Atmospheric Events  *(§6.4)*

| Event | Definition |
|-------|-----------|
| E9 | Pressure tier: campaign quartiles — captures seasonal weather variation |
| E10 | Pressure drop: delta < −0.5 hPa per 60s — HVAC switch or door opening |
| E11 | Humidity tier: dry (<40%) / normal (40–55%) / humid (55–70%) / very_humid (>70%) |
| E12 | Temperature tier: campaign quartiles — HVAC set-point shifts seasonally |

Note: pressure values in the dataset are scaled (287–348 range) rather than true hPa.  
All six devices show identical scaling — relative variation is preserved and analysis is unaffected.

In [6]:
# E9: pressure tier — quartile-based to capture seasonal variation
df['e9_pressure_tier'] = pd.qcut(
    df['pressure'], q=4,
    labels=['low', 'medium_low', 'medium_high', 'high'],
    duplicates='drop'
)

# E10: pressure drop — HVAC switch or door-opening transient
df['e10_pressure_drop'] = (
    df.groupby('device_id')['pressure'].diff() < -0.5
).astype(int)

# E11: humidity tier — absolute bands
df['e11_humidity_tier'] = pd.cut(
    df['humidity'],
    bins=[0, 40, 55, 70, 101],
    labels=['dry', 'normal', 'humid', 'very_humid']
)

# E12: temperature tier — quartile-based because HVAC set-point shifts across seasons
df['e12_temp_tier'] = pd.qcut(
    df['temperature'], q=4,
    labels=['cold', 'cool', 'warm', 'hot'],
    duplicates='drop'
)

print(f'Pressure range   : {df["pressure"].min():.1f} – {df["pressure"].max():.1f} (scaled)')
print(f'Humidity range   : {df["humidity"].min():.1f} – {df["humidity"].max():.1f} %')
print(f'Temperature range: {df["temperature"].min():.1f} – {df["temperature"].max():.1f} °C')
print(f'Pressure drops   : {df["e10_pressure_drop"].sum():,} intervals')

Pressure range   : 286.9 – 347.6 (scaled)
Humidity range   : 14.0 – 60.2 %
Temperature range: 13.7 – 44.0 °C
Pressure drops   : 160 intervals


## 5 · Transmission Configuration Events  *(§6.5)*

SF directly controls Time-on-Air (ToA) — the duration a packet occupies the channel.  
Under LoRaWAN's ALOHA protocol, longer ToA = wider collision window.

ADR was disabled and SFs were manually rotated across the campaign.  
SF distribution is approximately equal (~25% each) — ensuring fair event comparison.

| SF | ToA (ms) | Collision window vs SF7 |
|----|---------|------------------------|
| 7 | 71.9 | baseline |
| 8 | 133.6 | 1.9× |
| 9 | 246.8 | 3.4× |
| 10 | 452.6 | 6.3× |

| Event | Definition |
|-------|-----------|
| E13 | SF categorical: 7 / 8 / 9 / 10 |
| E14 | SF tier: low_sf (7–8) / high_sf (9–10) |
| E15 | ToA class: short_toa / long_toa — collision risk expressed explicitly |

In [7]:
TOA_MS = {7: 71.9, 8: 133.6, 9: 246.8, 10: 452.6}   # ms at BW=125kHz, 26-byte payload

# E13: spreading factor — categorical
df['e13_sf'] = df['SF'].astype(int)

# E14: SF tier — binary split at the ToA doubling boundary (SF8→SF9)
df['e14_sf_tier'] = df['e13_sf'].apply(
    lambda s: 'low_sf' if s in (7, 8) else 'high_sf'
)

# E15: ToA class and numeric value — kept for continuous analysis in notebook 03
df['e15_toa_ms']    = df['e13_sf'].map(TOA_MS)
df['e15_toa_class'] = df['e14_sf_tier'].map({'low_sf': 'short_toa', 'high_sf': 'long_toa'})

print('SF distribution:')
print(df['e13_sf'].value_counts(normalize=True).mul(100).sort_index().round(1).astype(str).add('%').to_string())

SF distribution:
e13_sf
7     26.0%
8     25.8%
9     24.7%
10    23.4%


## 6 · Burst Loss Event  *(§6.6)*

Classifies each interval by how many consecutive packets were lost before the received packet.  
Burst loss is qualitatively different from isolated loss — it creates contiguous data gaps  
that cannot be recovered by interpolation.

**B=3 threshold:** 3 consecutive losses = 3-minute gap — minimum that visibly disrupts  
slowly-varying environmental signals like CO₂ and temperature.

SF rotation artifacts are set to 0 before classification — they are deterministic transition  
losses already flagged by `is_sf_artifact` and not environmental bursts.

| Class | Condition | Interpretation |
|-------|-----------|---------------|
| no_loss | 0 | Clean reception |
| isolated | 1 | Single random collision |
| small_burst | 2–4 | Brief channel degradation |
| large_burst | 5+ | Sustained interference |

In [8]:
# E16: burst loss classification
# SF artifacts replaced with 0 — deterministic transition losses, not environmental bursts
df['e16_loss_type'] = pd.cut(
    df['mac_to_radio_loss'].where(~df['is_sf_artifact'], 0),
    bins=[-1, 0, 1, 4, 10_000_000],
    labels=['no_loss', 'isolated', 'small_burst', 'large_burst']
)

print('Loss type distribution:')
print(df['e16_loss_type'].value_counts(normalize=True).mul(100).round(1).astype(str).add('%').to_string())

Loss type distribution:
e16_loss_type
no_loss        98.2%
isolated        1.2%
small_burst     0.4%
large_burst     0.2%


## 7 · Signal Context Events  *(§6.7)*

RSSI and ESP are only available for received packets — never for lost ones.  
E17 and E18 use the signal quality of the packet received immediately before the loss interval  
as a proxy for channel state at the time of loss.

| Event | Definition |
|-------|-----------|
| E17 | RSSI tier: weak (<−90) / moderate (−90 to −70) / strong (>−70 dBm) |
| E18 | ESP tier: campaign tertile-based — ESP combines RSSI and SNR (§2.2.3) |

In [9]:
# E17: RSSI tier — thresholds reflect SF7-SF10 receiver sensitivity range (Table 2.2)
df['e17_rssi_tier'] = pd.cut(
    df['rssi'],
    bins=[-200, -90, -70, 0],
    labels=['weak', 'moderate', 'strong']
)

# E18: ESP tier — tertile-based, no obvious physical breakpoints in ESP distribution
df['e18_esp_tier'] = pd.qcut(
    df['esp'], q=3,
    labels=['low_esp', 'medium_esp', 'high_esp'],
    duplicates='drop'
)

print('RSSI tier distribution:')
print(df['e17_rssi_tier'].value_counts(normalize=True).mul(100).round(1).astype(str).add('%').to_string())
print(f'\nRSSI range: {df["rssi"].min():.0f} to {df["rssi"].max():.0f} dBm')
print(f'ESP  range: {df["esp"].min():.1f} to {df["esp"].max():.1f} dBm')

RSSI tier distribution:
e17_rssi_tier
strong      51.0%
moderate    28.9%
weak        20.1%

RSSI range: -128 to -29 dBm
ESP  range: -141.1 to -29.8 dBm


## 8 · Event Taxonomy  *(Table 6.1)*

In [10]:
event_taxonomy = [
    ('E1',  'Occupancy',      'e1_co2_tier',        'CO2 concentration tier'),
    ('E2',  'Occupancy',      'e2_is_weekday',       'Weekday vs weekend'),
    ('E3',  'Occupancy',      'e3_co2_rising',       'CO2 rising > 20ppm/60s'),
    ('E4',  'Temporal',       'e4_time_of_day',      'Time of day band'),
    ('E5',  'Temporal',       'e5_office_hours',     'Office hours'),
    ('E6',  'Temporal',       'e6_season',           'Season'),
    ('E7',  'Air Quality',    'e7_pm25_spike',       'PM2.5 device-specific spike'),
    ('E8',  'Air Quality',    'e8_pm25_tier',        'PM2.5 absolute tier'),
    ('E9',  'Atmospheric',    'e9_pressure_tier',    'Pressure tier'),
    ('E10', 'Atmospheric',    'e10_pressure_drop',   'Pressure drop event'),
    ('E11', 'Atmospheric',    'e11_humidity_tier',   'Humidity tier'),
    ('E12', 'Atmospheric',    'e12_temp_tier',       'Temperature tier'),
    ('E13', 'TX Config',      'e13_sf',              'Spreading factor'),
    ('E14', 'TX Config',      'e14_sf_tier',         'SF tier (low/high)'),
    ('E15', 'TX Config',      'e15_toa_class',       'Time-on-Air class'),
    ('E16', 'Burst Loss',     'e16_loss_type',       'Loss episode type'),
    ('E17', 'Signal Context', 'e17_rssi_tier',       'RSSI tier'),
    ('E18', 'Signal Context', 'e18_esp_tier',        'ESP tier'),
]

print(f"{'ID':<5} {'Family':<16} {'Column':<22} {'Name':<30} {'Possible Values'}")
print('-' * 105)
for eid, family, col, name in event_taxonomy:
    dtype = df[col].dtype
    if col == 'e13_sf':
        coverage = f'values: {sorted(df[col].unique().tolist())}'
    elif str(dtype) in ['int64', 'bool']:
        coverage = f'{df[col].mean()*100:.1f}% active'
    else:
        coverage = str(sorted([x for x in df[col].unique() if str(x) != 'nan']))
    print(f'{eid:<5} {family:<16} {col:<22} {name:<30} {coverage}')

ID    Family           Column                 Name                           Possible Values
---------------------------------------------------------------------------------------------------------
E1    Occupancy        e1_co2_tier            CO2 concentration tier         ['background', 'high', 'moderate']
E2    Occupancy        e2_is_weekday          Weekday vs weekend             70.9% active
E3    Occupancy        e3_co2_rising          CO2 rising > 20ppm/60s         0.3% active
E4    Temporal         e4_time_of_day         Time of day band               ['evening', 'morning', 'night', 'peak']
E5    Temporal         e5_office_hours        Office hours                   29.2% active
E6    Temporal         e6_season              Season                         ['autumn', 'spring', 'winter']
E7    Air Quality      e7_pm25_spike          PM2.5 device-specific spike    10.0% active
E8    Air Quality      e8_pm25_tier           PM2.5 absolute tier            ['clean', 'elevated', 'moder

## 9 · Save

In [11]:
original_cols = [c for c in df.columns
                 if not (c.startswith('e') and '_' in c)
                 and c not in ['hour', 'dayofweek', 'month', 'exp_pl']]

event_cols_ordered = [
    'e1_co2_tier',    'e2_is_weekday',  'e3_co2_rising',
    'e4_time_of_day', 'e5_office_hours','e6_season',
    'e7_pm25_spike',  'e8_pm25_tier',
    'e9_pressure_tier','e10_pressure_drop','e11_humidity_tier','e12_temp_tier',
    'e13_sf',         'e14_sf_tier',    'e15_toa_ms',       'e15_toa_class',
    'e16_loss_type',  'e17_rssi_tier',  'e18_esp_tier'
]

df = df[[c for c in original_cols if c in df.columns] + event_cols_ordered]
df.to_csv(DATA_DIR / '2_events.csv', index=False)

print(f'Saved  : {DATA_DIR / "2_events.csv"}')
print(f'Rows   : {len(df):,}')
print(f'Columns: {len(df.columns)} total | 18 events + e15_toa_ms')

Saved  : ..\data\2_events.csv
Rows   : 1,217,313
Columns: 46 total | 18 events + e15_toa_ms
